# TEK spike — a temporary Linux VM on Kaggle

**Goal:** prove the whole chain in one cell: QEMU boots a real Linux VM inside a Kaggle
CPU notebook, noVNC serves its screen, and a Cloudflare quick tunnel gives your browser
a URL to reach it. Your machine only draws the screen and sends keyboard/mouse.

**Before you run**

1. Create a **Kaggle Notebook** (Python, CPU — no GPU needed).
2. In *Settings → Internet*, turn **Internet on**. The cell needs it for apt, the ISO,
   and cloudflared.
3. Paste this notebook (or import the `.ipynb`), then **Run all**.
4. First run takes ~5–8 minutes. When it prints `OPEN THIS IN YOUR BROWSER`, open that
   `https://….trycloudflare.com` link, type the VNC password, and log in to Alpine with
   user `root` (no password).

**What this spike answers**

- Is `/dev/kvm` exposed? (Expected: no → QEMU runs under TCG software emulation.)
- How slow is a TCG guest on Kaggle's 4 vCPUs?
- Does an outbound Cloudflare tunnel work from a Kaggle notebook?
- Does noVNC render and accept input through it?

Keep the session open while you use the VM. Everything dies with the session — that is
the point of TEK.


In [ ]:
# TEK spike: boot Alpine in QEMU on Kaggle, reach it from your browser.
# Requires: Kaggle notebook, Internet enabled. One cell, ~5-8 min first run.
import os
import re
import secrets
import shutil
import socket
import subprocess
import time
import urllib.request

from IPython.display import display

T0 = time.time()
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
ISO = f"{TMP}/alpine-virt.iso"
MON = f"{TMP}/qemu-mon"
CF_LOG = f"{TMP}/cloudflared.log"


def run(cmd, check=True, quiet=False):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    if out and not quiet:
        print(out[-3000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p


def mon(cmd):
    s = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
    s.settimeout(5)
    s.connect(MON)
    s.sendall((cmd + "\n").encode())
    time.sleep(0.3)
    try:
        data = s.recv(65536).decode(errors="replace")
    except socket.timeout:
        data = ""
    s.close()
    return data.strip()


print("== 1. host facts ==")
run("nproc; grep -m1 'model name' /proc/cpuinfo; free -g | head -2; df -h /kaggle/temp 2>/dev/null | tail -1", check=False)
print("KVM available:", os.path.exists("/dev/kvm"), "-> QEMU will use TCG if False")

print("\n== 2. install QEMU, noVNC, cloudflared ==")
run("apt-get -qq update")
run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y qemu-system-x86 qemu-utils novnc websockify")
if not shutil.which("cloudflared"):
    run("curl -fsSL -o /usr/local/bin/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.chmod("/usr/local/bin/cloudflared", 0o755)

print("\n== 3. fetch a tiny Linux ISO ==")
if not os.path.exists(ISO):
    base = "https://dl-cdn.alpinelinux.org/alpine/latest-stable/releases/x86_64/"
    html = urllib.request.urlopen(base, timeout=60).read().decode()
    names = re.findall(r'href="(alpine-virt-[\d.]+-x86_64\.iso)"', html)
    if not names:
        raise RuntimeError("no Alpine virt ISO found in the mirror listing")
    name = max(names, key=lambda n: tuple(int(x) for x in re.findall(r"\d+", n)[:3]))
    print("picked:", name)
    run(f"curl -fL --retry 3 -o {ISO} {base}{name}")
print("ISO:", ISO, os.path.getsize(ISO) // (1024 * 1024), "MB")

print("\n== 4. boot the VM (TCG, 4 vCPU, 4 GB) ==")
run("pkill -f qemu-system-x86_64 || true; pkill -f websockify || true; pkill -f cloudflared || true", check=False)
vnc_pass = secrets.token_urlsafe(6)
qemu = (
    "qemu-system-x86_64 -accel tcg,thread=multi -cpu max -smp 4 -m 4096 "
    f"-cdrom {ISO} -boot d -vga std -vnc 127.0.0.1:0,password=on "
    f"-monitor unix:{MON},server,nowait -daemonize"
)
run(qemu)
for _ in range(20):
    if os.path.exists(MON):
        break
    time.sleep(0.5)
mon(f"change vnc password {vnc_pass}")
status = mon("info status")
print("QEMU:", status.splitlines()[0] if status else "no monitor reply")

print("\n== 5. serve noVNC ==")
run("pkill -f websockify || true", check=False)
subprocess.Popen(
    ["websockify", "--web=/usr/share/novnc", "6080", "127.0.0.1:5900"],
    stdout=open(f"{TMP}/websockify.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(2)
print("noVNC local check:", urllib.request.urlopen("http://127.0.0.1:6080/vnc.html", timeout=10).status)

print("\n== 6. open a Cloudflare tunnel ==")


def start_tunnel(proto=None):
    if os.path.exists(CF_LOG):
        os.remove(CF_LOG)
    cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:6080",
           "--no-autoupdate", "--logfile", CF_LOG]
    if proto:
        cmd += ["--protocol", proto]
    subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(45):
        time.sleep(1)
        txt = open(CF_LOG, errors="replace").read() if os.path.exists(CF_LOG) else ""
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
        if m:
            return m.group(0)
    return None


url = start_tunnel() or start_tunnel("http2")
if not url:
    run(f"tail -40 {CF_LOG}", check=False)
    raise RuntimeError("cloudflared did not produce a URL")

print("\n" + "=" * 64)
print("OPEN THIS IN YOUR BROWSER:", url)
print("VNC password:", vnc_pass)
print("Alpine login: root   (no password)")
print("=" * 64)

print("\n== 7. wait for the guest to draw, then screenshot ==")
shot = f"{TMP}/screen.ppm"
img = None
for _ in range(12):
    time.sleep(10)
    mon(f"screendump {shot}")
    if os.path.exists(shot):
        try:
            from PIL import Image, ImageStat
            candidate = Image.open(shot).convert("RGB")
            if ImageStat.Stat(candidate.convert("L")).stddev[0] > 5:
                img = candidate
                break
        except Exception:
            pass
if img is not None:
    img.thumbnail((900, 900))
    display(img)
    print("Screenshot captured — the guest is drawing.")
else:
    print("No non-blank frame yet. Give it another minute, then reload the tunnel URL.")

print(f"\nDone in {time.time() - T0:.0f}s. The VM and tunnel live as long as this session does.")
